In [4]:
import numpy as np
import gymnasium as gym
import matplotlib.pyplot as plt
from tile_coder import TileCoder
from tqdm import tqdm

In [2]:
env = gym.make("MountainCar-v0")
print(env.observation_space)
coder = TileCoder(
    low=[-1.2, -0.07],
    high=[0.6, 0.07],
    num_tiles=8,
    tile_per_dimension=[8, 8],
    num_actions=3
)

Box([-1.2  -0.07], [0.6  0.07], (2,), float32)


In [3]:
def epsilon_greedy(w:np.ndarray, state, coder:TileCoder, epsilon:0.1):
    if np.random.rand() < epsilon:
        return np.random.choice(coder.num_actions)

    values = []
    for i in range(coder.num_actions):
        values.append(w.T@coder.get_vector(state, i))

    return int(np.argmax(values))

In [9]:
def true_online_sarsa_lambda(
    w:np.ndarray,
    env:gym.Env,
    coder:TileCoder,
    episodes:int,
    alpha:float=0.1,
    gamma:float=0.99,
    lambda_:float=0.1,
    epsilon:float=0.1
):
    rewards_per_episiode = []

    for episode in tqdm(range(episodes)):
        state = env.reset()[0]
        action = epsilon_greedy(w, state, coder, epsilon)
        x = coder.get_vector(state, action)

        sum_rewards = 0

        old_q = 0
        z = np.zeros_like(w)
        terminated = False
        while not terminated:
            next_state, reward, terminated, _, _ = env.step(action)
            sum_rewards += reward
            next_action = epsilon_greedy(w, next_state, coder, epsilon)
            x_prime = coder.get_vector(next_state, next_action)
            q = w.T@x
            q_prime = w.T@x_prime

            delta = reward + gamma * q_prime - q
            z = gamma * lambda_ * z + (1 - gamma * lambda_ * alpha * z.T@x) * x
            w = w + alpha * (delta + q - old_q) * z + alpha * (old_q - q) * x
            old_q = q_prime
            x = x_prime
            state = next_state
            action = next_action
        rewards_per_episiode.append(sum_rewards)

    return w, rewards_per_episiode

In [10]:
w = np.zeros((coder.num_features,), dtype=np.float32)
w, rewards_per_episode = true_online_sarsa_lambda(
    w=w, 
    env=env,
    coder=coder,
    episodes=500,
    alpha=0.1/8,
    gamma=0.99,
    lambda_=0.9,
    epsilon=0.1
)

100%|██████████| 500/500 [01:23<00:00,  5.97it/s]


In [11]:
# run infer
from gymnasium.wrappers import RecordVideo
eval_env = gym.make("MountainCar-v0", render_mode='rgb_array')
eval_env = RecordVideo(
    eval_env,
    "videos/",
    episode_trigger=lambda eps: True
)
state = eval_env.reset()[0]
terminated = False
steps = 0

while not terminated:
    print(f"\rStep {steps+1:<5}", end="")
    action = epsilon_greedy(w, state, coder, 0.0)
    state, _, terminated, _, _ = eval_env.step(action)
    steps += 1

print("agent took", steps, "steps")
eval_env.close()

/home/alireza/miniconda3/envs/torch/lib/python3.13/site-packages/gymnasium/wrappers/rendering.py:283: UserWarning: WARN: Overwriting existing videos at /home/alireza/Desktop/Reinforcement_Learning/Chapter 12/videos folder (try specifying a different `video_folder` for the `RecordVideo` wrapper if this is not desired)
  logger.warn(


Step 111  agent took 111 steps
